# RV32I 5-Stage Pipeline CPU — PYNQ-ZU 板上整合測試（RV32I + Zbb + MUL + illegal）
依序執行所有 cell。重新上傳 bitstream 後請先 **Kernel → Restart**。

- T1~T3：reset / run / 重跑
- T4~T6：trace buffer 逐 cycle 對照 ISA 黃金模型
- T7：Zbb 17 條指令與 RV32M MUL 逐筆驗證
- T8：illegal 偵測（需含 illegal 暫存器的 bitstream）

In [ ]:
# ① 載入 Overlay 與 MMIO（重新上傳 bitstream 後請先 Kernel → Restart，避免 PYNQ 使用舊 bitstream 快取）
import time, json
from pynq import Overlay, MMIO

BIT  = "/home/xilinx/jupyter_notebooks/design_1_wrapper.bit"   # 同資料夾需有同名 .hwh
BASE = 0x80000000
SIZE = 0x10000

ol   = Overlay(BIT)
mmio = MMIO(BASE, SIZE)
print("CPU IP :", ol.ip_dict["rv32i_pipelined_cpu_0"]["type"])


In [ ]:
# ② 黃金模型（RV32I + Zbb + MUL + illegal 規則）與上板測試程式
# 這份 PROGRAM 必須與 imem.v 內建程式（= program.hex）完全相同。
"""
rv32i_model.py  RV32I + Zbb + MUL 的 Python 參考模型（golden model）與迷你組譯器。

用途
  * 提供 Zbb 每個指令的「標準答案」(zbb_ref)
  * 提供 ISA 層級的程式執行模型 run()，輸出最終暫存器、記憶體與寫回序列
  * 提供指令編碼器 enc_*，產生測試程式的機器碼（已對照 GNU binutils 驗證，見 verify_encodings.py）

模型與硬體 control_unit.v 的 illegal 規則一致：不支援的指令視為 NOP 並被記錄。
本核心目前只支援 lw/sw（沒有 lb/lh/sb/sh）、RV32M 只有 MUL（MULH*/DIV*/REM* 仍為 illegal）、沒有 CSR / FENCE / ECALL。
"""
M32 = 0xFFFFFFFF


def s32(v):
    v &= M32
    return v - (1 << 32) if v & 0x80000000 else v


def sx(v, bits):
    v &= (1 << bits) - 1
    return v - (1 << bits) if v & (1 << (bits - 1)) else v


# ----------------------------------------------------------------------------
# Zbb 標準答案
# ----------------------------------------------------------------------------
def _clz(a):
    return 32 - a.bit_length()


def _ctz(a):
    return 32 if a == 0 else (a & -a).bit_length() - 1


def _rol(a, n):
    n &= 31
    return ((a << n) | (a >> (32 - n))) & M32 if n else a


def _ror(a, n):
    n &= 31
    return ((a >> n) | (a << (32 - n))) & M32 if n else a


ZBB_REF = {
    "andn":   lambda a, b: a & ~b & M32,
    "orn":    lambda a, b: (a | ~b) & M32,
    "xnor":   lambda a, b: ~(a ^ b) & M32,
    "min":    lambda a, b: a if s32(a) < s32(b) else b,
    "minu":   lambda a, b: a if a < b else b,
    "max":    lambda a, b: b if s32(a) < s32(b) else a,
    "maxu":   lambda a, b: b if a < b else a,
    "rol":    lambda a, b: _rol(a, b),
    "ror":    lambda a, b: _ror(a, b),
    "clz":    lambda a, b: _clz(a),
    "ctz":    lambda a, b: _ctz(a),
    "cpop":   lambda a, b: bin(a).count("1"),
    "sext_b": lambda a, b: sx(a, 8) & M32,
    "sext_h": lambda a, b: sx(a, 16) & M32,
    "zext_h": lambda a, b: a & 0xFFFF,
    "rev8":   lambda a, b: int.from_bytes(a.to_bytes(4, "little"), "big"),
    "orc_b":  lambda a, b: sum(0xFF << (8 * i) for i in range(4) if (a >> (8 * i)) & 0xFF),
}

# 與 zbb_unit.v / control_unit.v 的 op 編碼順序一致
ZBB_OPS = ["andn", "orn", "xnor", "min", "minu", "max", "maxu", "rol", "ror",
           "clz", "ctz", "cpop", "sext_b", "sext_h", "zext_h", "rev8", "orc_b"]
ZBB_BINARY = ["andn", "orn", "xnor", "min", "minu", "max", "maxu", "rol", "ror"]   # R-type, 用 rs2
ZBB_UNARY = ["clz", "ctz", "cpop", "sext_b", "sext_h", "zext_h", "rev8", "orc_b"]  # 只用 rs1

# ----------------------------------------------------------------------------
# 編碼器
# ----------------------------------------------------------------------------
OP, OPIMM, LOAD, STORE, BRANCH, JAL, JALR, LUI, AUIPC = (
    0b0110011, 0b0010011, 0b0000011, 0b0100011, 0b1100011, 0b1101111, 0b1100111, 0b0110111, 0b0010111)


def enc_r(op, f3, f7, rd, rs1, rs2):
    return (f7 << 25) | (rs2 << 20) | (rs1 << 15) | (f3 << 12) | (rd << 7) | op


def enc_i(op, f3, rd, rs1, imm):
    return ((imm & 0xFFF) << 20) | (rs1 << 15) | (f3 << 12) | (rd << 7) | op


def enc_s(f3, rs1, rs2, imm):
    return (((imm >> 5) & 0x7F) << 25) | (rs2 << 20) | (rs1 << 15) | (f3 << 12) | ((imm & 0x1F) << 7) | STORE


def enc_b(f3, rs1, rs2, imm):
    return (((imm >> 12) & 1) << 31) | (((imm >> 5) & 0x3F) << 25) | (rs2 << 20) | (rs1 << 15) | (f3 << 12) | \
           (((imm >> 1) & 0xF) << 8) | (((imm >> 11) & 1) << 7) | BRANCH


def enc_u(op, rd, imm20):
    return ((imm20 & 0xFFFFF) << 12) | (rd << 7) | op


def enc_j(rd, imm):
    return (((imm >> 20) & 1) << 31) | (((imm >> 1) & 0x3FF) << 21) | (((imm >> 11) & 1) << 20) | \
           (((imm >> 12) & 0xFF) << 12) | (rd << 7) | JAL


BASE_R = {"add": (0, 0x00), "sub": (0, 0x20), "sll": (1, 0), "slt": (2, 0), "sltu": (3, 0),
          "xor": (4, 0), "srl": (5, 0), "sra": (5, 0x20), "or": (6, 0), "and": (7, 0)}
M_R = {"mul": (0, 0x01)}          # RV32M：只實作 MUL（低 32 位元）
BASE_I = {"addi": 0, "slti": 2, "sltiu": 3, "xori": 4, "ori": 6, "andi": 7}
SHIFT_I = {"slli": (1, 0x00), "srli": (5, 0x00), "srai": (5, 0x20), "rori": (5, 0x30)}
ZBB_R = {"andn": (7, 0x20), "orn": (6, 0x20), "xnor": (4, 0x20),
         "min": (4, 0x05), "minu": (5, 0x05), "max": (6, 0x05), "maxu": (7, 0x05),
         "rol": (1, 0x30), "ror": (5, 0x30)}
# 單運算元：(opcode, funct3, funct7, rs2 欄位固定值)
ZBB_U = {"clz": (OPIMM, 1, 0x30, 0), "ctz": (OPIMM, 1, 0x30, 1), "cpop": (OPIMM, 1, 0x30, 2),
         "sext_b": (OPIMM, 1, 0x30, 4), "sext_h": (OPIMM, 1, 0x30, 5),
         "rev8": (OPIMM, 5, 0x34, 0x18), "orc_b": (OPIMM, 5, 0x14, 0x07),
         "zext_h": (OP, 4, 0x04, 0)}


def asm(name, *a):
    """asm('add', rd, rs1, rs2) / asm('addi', rd, rs1, imm) / asm('clz', rd, rs1) / asm('lw', rd, rs1, off) /
       asm('sw', rs2, rs1, off) / asm('beq', rs1, rs2, off) / asm('lui', rd, imm20) / asm('jal', rd, off)"""
    if name in BASE_R:
        f3, f7 = BASE_R[name]; return enc_r(OP, f3, f7, *a)
    if name in ZBB_R:
        f3, f7 = ZBB_R[name]; return enc_r(OP, f3, f7, *a)
    if name in M_R:
        f3, f7 = M_R[name]; return enc_r(OP, f3, f7, *a)
    if name in BASE_I:
        return enc_i(OPIMM, BASE_I[name], *a)
    if name in SHIFT_I:
        f3, f7 = SHIFT_I[name]; rd, rs1, sh = a
        return enc_i(OPIMM, f3, rd, rs1, (f7 << 5) | (sh & 31))
    if name in ZBB_U:
        op, f3, f7, r2 = ZBB_U[name]; rd, rs1 = a
        return enc_r(op, f3, f7, rd, rs1, r2)
    if name == "lw":
        rd, rs1, off = a; return enc_i(LOAD, 2, rd, rs1, off)
    if name == "sw":
        rs2, rs1, off = a; return enc_s(2, rs1, rs2, off)
    if name in ("beq", "bne", "blt", "bge", "bltu", "bgeu"):
        f3 = {"beq": 0, "bne": 1, "blt": 4, "bge": 5, "bltu": 6, "bgeu": 7}[name]
        return enc_b(f3, a[0], a[1], a[2])
    if name == "lui":
        return enc_u(LUI, *a)
    if name == "auipc":
        return enc_u(AUIPC, *a)
    if name == "jal":
        return enc_j(*a)
    if name == "jalr":
        rd, rs1, off = a; return enc_i(JALR, 0, rd, rs1, off)
    raise ValueError(name)


def li(rd, value):
    """載入 32-bit 常數 → [lui, addi]"""
    value &= M32
    lo = sx(value, 12)
    hi = ((value - lo) & M32) >> 12
    return [asm("lui", rd, hi), asm("addi", rd, rd, lo)]


# ----------------------------------------------------------------------------
# 解碼 + 執行（鏡像 control_unit.v）
# ----------------------------------------------------------------------------
def decode(ins):
    """回傳 ('name', fields...) 或 None（= illegal，硬體當 NOP）"""
    op = ins & 0x7F
    rd = (ins >> 7) & 31; f3 = (ins >> 12) & 7
    rs1 = (ins >> 15) & 31; rs2 = (ins >> 20) & 31; f7 = ins >> 25
    immi = sx(ins >> 20, 12)
    if op == OP:
        for n, (a, b) in BASE_R.items():
            if (f3, f7) == (a, b): return (n, rd, rs1, rs2)
        for n, (a, b) in ZBB_R.items():
            if (f3, f7) == (a, b): return (n, rd, rs1, rs2)
        for n, (a, b) in M_R.items():
            if (f3, f7) == (a, b): return (n, rd, rs1, rs2)
        if (f3, f7) == (4, 0x04) and rs2 == 0: return ("zext_h", rd, rs1, 0)
        return None
    if op == OPIMM:
        for n, a in BASE_I.items():
            if f3 == a: return (n, rd, rs1, immi)
        if f3 == 1:
            if f7 == 0: return ("slli", rd, rs1, rs2)
            if f7 == 0x30:
                m = {0: "clz", 1: "ctz", 2: "cpop", 4: "sext_b", 5: "sext_h"}
                return (m[rs2], rd, rs1, 0) if rs2 in m else None
            return None
        if f3 == 5:
            if f7 == 0: return ("srli", rd, rs1, rs2)
            if f7 == 0x20: return ("srai", rd, rs1, rs2)
            if f7 == 0x30: return ("rori", rd, rs1, rs2)
            if f7 == 0x34 and rs2 == 0x18: return ("rev8", rd, rs1, 0)
            if f7 == 0x14 and rs2 == 0x07: return ("orc_b", rd, rs1, 0)
            return None
        return None
    if op == LOAD:
        return ("lw", rd, rs1, immi) if f3 == 2 else None
    if op == STORE:
        imm = sx(((ins >> 25) << 5) | ((ins >> 7) & 31), 12)
        return ("sw", rs2, rs1, imm) if f3 == 2 else None
    if op == BRANCH:
        if f3 in (2, 3): return None
        imm = sx(((ins >> 31) << 12) | (((ins >> 7) & 1) << 11) | (((ins >> 25) & 0x3F) << 5) | (((ins >> 8) & 0xF) << 1), 13)
        return ({0: "beq", 1: "bne", 4: "blt", 5: "bge", 6: "bltu", 7: "bgeu"}[f3], rs1, rs2, imm)
    if op == JAL:
        imm = sx(((ins >> 31) << 20) | (((ins >> 12) & 0xFF) << 12) | (((ins >> 20) & 1) << 11) | (((ins >> 21) & 0x3FF) << 1), 21)
        return ("jal", rd, imm)
    if op == JALR:
        return ("jalr", rd, rs1, immi) if f3 == 0 else None
    if op == LUI:
        return ("lui", rd, ins & 0xFFFFF000)
    if op == AUIPC:
        return ("auipc", rd, ins & 0xFFFFF000)
    return None


def _base_r(n, a, b):
    return {"add": a + b, "sub": a - b, "sll": a << (b & 31), "slt": int(s32(a) < s32(b)), "sltu": int(a < b),
            "xor": a ^ b, "srl": a >> (b & 31), "sra": s32(a) >> (b & 31), "or": a | b, "and": a & b}[n] & M32


def run(prog, max_steps=50000, mem_words=1024):   # dmem.v: 4096 位元組 = 1024 字（索引 addr[11:2]）
    """執行到「自我跳躍」(spin) 為止。
    回傳 dict: regs[32], mem{word_index: value}, wb[(rd,val)...] (rd!=0, 程式順序), illegal[pc...], pc"""
    x = [0] * 32
    mem = {}
    wb, illegal, winfo = [], [], []
    pc = 0
    for _ in range(max_steps):
        ins = prog[pc >> 2] if (pc >> 2) < len(prog) else 0x00000013       # imem 以 NOP 填滿
        d = decode(ins)
        npc = (pc + 4) & M32
        val = None
        rd = None
        if d is None:
            illegal.append(pc)
        else:
            n = d[0]
            if n in BASE_R or n in ZBB_R or n in M_R:
                _, rd, r1, r2 = d
                if n in M_R: val = (x[r1] * x[r2]) & M32
                elif n in BASE_R: val = _base_r(n, x[r1], x[r2])
                else: val = ZBB_REF[n](x[r1], x[r2])
            elif n in BASE_I:
                _, rd, r1, imm = d
                a = x[r1]; b = imm & M32
                val = {"addi": a + b, "slti": int(s32(a) < imm), "sltiu": int(a < b),
                       "xori": a ^ b, "ori": a | b, "andi": a & b}[n] & M32
            elif n in ("slli", "srli", "srai", "rori"):
                _, rd, r1, sh = d
                a = x[r1]
                val = {"slli": a << sh, "srli": a >> sh, "srai": s32(a) >> sh, "rori": _ror(a, sh)}[n] & M32
            elif n in ZBB_UNARY:
                _, rd, r1, _ = d
                val = ZBB_REF[n](x[r1], 0)
            elif n == "lw":
                _, rd, r1, imm = d
                val = mem.get(((x[r1] + imm) & M32) >> 2 & (mem_words - 1), 0)
            elif n == "sw":
                _, r2, r1, imm = d
                mem[((x[r1] + imm) & M32) >> 2 & (mem_words - 1)] = x[r2]
            elif n in ("beq", "bne", "blt", "bge", "bltu", "bgeu"):
                _, r1, r2, imm = d
                a, b = x[r1], x[r2]
                t = {"beq": a == b, "bne": a != b, "blt": s32(a) < s32(b), "bge": s32(a) >= s32(b),
                     "bltu": a < b, "bgeu": a >= b}[n]
                if t: npc = (pc + imm) & M32
            elif n == "jal":
                _, rd, imm = d; val = (pc + 4) & M32; npc = (pc + imm) & M32
            elif n == "jalr":
                _, rd, r1, imm = d; val = (pc + 4) & M32; npc = (x[r1] + imm) & ~1 & M32
            elif n == "lui":
                _, rd, imm = d; val = imm
            elif n == "auipc":
                _, rd, imm = d; val = (pc + imm) & M32
        if val is not None and rd:
            x[rd] = val & M32
            wb.append((rd, val & M32))
            winfo.append((pc, d[0], rd, val & M32))      # (指令 PC, 助記符, rd, 值)
        if npc == pc:
            break
        pc = npc
    return {"regs": x, "mem": mem, "wb": wb, "winfo": winfo, "illegal": illegal, "pc": pc}


PROGRAM = [
    0x00500093, 0x00a00113, 0x002081b3, 0x00500213,
    0x00320293, 0x00528b93, 0x00100313, 0x00130393,
    0x00138413, 0x00000593, 0x02a00493, 0x0095a023,
    0x0005a503, 0x00950633, 0x00000493, 0x00050493,
    0x00000693, 0x00100793, 0x00100813, 0x01078463,
    0x06300693, 0x00700713, 0x00100913, 0x00200993,
    0x01391663, 0x00500893, 0x00900893, 0x00900893,
    0x00000013, 0x00000013, 0x00000a93, 0x00c000ef,
    0x00500a93, 0x00700a93, 0x06300a13, 0x0080006f,
    0x00500a93, 0x12345b37, 0x0040006f, 0xf0f090b7,
    0xa8108093, 0x00ff0137, 0x00310113, 0x000121b7,
    0x30018193, 0x4020f2b3, 0x4020e333, 0x4020c3b3,
    0x0a20c433, 0x0a20d4b3, 0x0a20e533, 0x0a20f5b3,
    0x60209633, 0x6020d6b3, 0x60c0d713, 0x60019793,
    0x60119813, 0x60209893, 0x60409913, 0x60509993,
    0x0800ca33, 0x6980da93, 0x2871db13, 0x60001b93,
    0x60101c13, 0x60201c93, 0x60009d13, 0x60109d93,
    0x60019e13, 0x01ce0eb3, 0x6980df13, 0x001f4fb3,
    0x01502423, 0x00802283, 0x00528333, 0x60219393,
    0x00400413, 0x00839463, 0x05500493, 0x00838463,
    0x02a50533, 0x06600513, 0x022087b3, 0x02108833,
    0x020188b3, 0xfff00993, 0x03398a33, 0x02398ab3,
    0x80000e37, 0x00200e93, 0x03de0f33, 0x033e0fb3,
    0x02208b33, 0x016b0bb3, 0x036b0c33, 0x01602623,
    0x00c02c83, 0x023c8d33, 0x03de83b3, 0x00400413,
    0x00839463, 0x03c00493, 0x00838463, 0x03de8db3,
    0x02b00513, 0x02a59633, 0x00000283, 0x00000000,
    0x0ff0000f, 0x00000073, 0x00100023, 0x40109593,
    0x60309693, 0x07a00713, 0x0000006f
]
_ref = run(PROGRAM)
EXP_WB, EXP_REGS, EXP_END_PC = _ref["wb"], _ref["regs"], _ref["pc"]
EXP_INFO, EXP_ILLEGAL, EXP_MEM = _ref["winfo"], _ref["illegal"], _ref["mem"]
print("黃金模型：%d 字程式、%d 筆寫回、%d 條 illegal（第一條 PC=0x%X）、spin PC=0x%X" %
      (len(PROGRAM), len(EXP_WB), len(EXP_ILLEGAL), EXP_ILLEGAL[0] if EXP_ILLEGAL else 0, EXP_END_PC))


In [ ]:
# ③ 暫存器對應與測試工具
CTRL, PC, WBD, DMEM       = 0x00, 0x04, 0x08, 0x0C
T_STATUS, T_IDX, T_PC, T_WB, T_META = 0x10, 0x14, 0x18, 0x1C, 0x20   # trace buffer
R_ILL, R_ILLPC, R_FEAT    = 0x24, 0x28, 0x2C                          # illegal 回報、功能旗標

def cpu_reset():    mmio.write(CTRL, 0x0)
def cpu_run():      mmio.write(CTRL, 0x1)
def arm_and_run():  mmio.write(CTRL, 0x2); mmio.write(CTRL, 0x1)       # reset+arm → run

def read_trace(n=256):
    rows = []
    for i in range(n):
        mmio.write(T_IDX, i)
        pc, wd, meta = mmio.read(T_PC), mmio.read(T_WB), mmio.read(T_META)
        rows.append((i, pc, wd, (meta >> 1) & 31, meta & 1))     # (cycle, pc, wb_data, wb_rd, wb_we)
    return rows

def wb_events(rows):
    return [(r[3], r[2]) for r in rows if r[4] == 1 and r[3] != 0]

RESULTS = []
def check(name, ok, detail=""):
    RESULTS.append((name, bool(ok), detail))
    print(("[PASS] " if ok else "[FAIL] ") + name + (("  —  " + detail) if detail else ""))
    return ok


In [ ]:
# ④ 基本測試：reset / run / 重跑
SPIN_PCS = {EXP_END_PC, EXP_END_PC + 4, EXP_END_PC + 8}     # spin 迴圈附近的 PC（含 flush 時的預取）

cpu_reset(); time.sleep(0.001)
r0, pc0 = mmio.read(CTRL), mmio.read(PC)
check("T1 reset：rst_n=0 且 PC=0", (r0 & 1) == 0 and pc0 == 0, "reg0=0x%X pc=0x%X" % (r0, pc0))

cpu_run(); time.sleep(0.01)
r0 = mmio.read(CTRL)
pcs = sorted({mmio.read(PC) for _ in range(100)})
check("T2a run：rst_n 寫入成功（reg0[0]=1）", r0 & 1 == 1, "reg0=0x%X" % r0)
check("T2b run：PC 停在 spin 迴圈附近（0x%X 起）" % EXP_END_PC, set(pcs) <= SPIN_PCS and len(pcs) > 0,
      "PC 集合=" + str([hex(p) for p in pcs]))

cpu_reset(); time.sleep(0.001)
check("T3a 再次 reset：PC 回到 0", mmio.read(PC) == 0)
ok = True
for k in range(5):
    cpu_reset(); time.sleep(0.001); cpu_run(); time.sleep(0.005)
    ok &= {mmio.read(PC) for _ in range(30)} <= SPIN_PCS
check("T3b 重複 reset/run 5 次結果一致", ok)


In [ ]:
# ⑤ 逐 cycle 驗證（需要 trace buffer bitstream）
arm_and_run(); time.sleep(0.01)
st = mmio.read(T_STATUS)
HAS_TRACE = bool(st & 0x80000000)
rows1 = None
if not HAS_TRACE:
    print("[SKIP] 偵測不到 trace buffer（status=0x%X）：略過 T4~T7" % st)
else:
    check("T4a trace 已填滿 256 筆", (st & 0x1FF) == 256, "status=0x%X" % st)
    rows1 = read_trace()
    got = wb_events(rows1)
    first_bad = next((i for i, (e, g) in enumerate(zip(EXP_WB, got)) if e != g), None)
    check("T4b 寫回序列與黃金模型一致（%d 筆）" % len(EXP_WB), got == EXP_WB,
          "" if got == EXP_WB else "第 %s 筆不同：預期 %s 實際 %s；筆數 預期%d 實際%d" % (
              first_bad, EXP_WB[first_bad] if first_bad is not None and first_bad < len(EXP_WB) else None,
              got[first_bad] if first_bad is not None and first_bad < len(got) else None, len(EXP_WB), len(got)))
    check("T4c reset 放開後第 0 個 cycle PC=0", rows1[0][1] == 0, "pc[0]=0x%X" % rows1[0][1])
    pcs_all = [r[1] for r in rows1]
    check("T4d PC 皆 4-byte 對齊且不跑出程式範圍",
          all(p % 4 == 0 for p in pcs_all) and max(pcs_all) <= EXP_END_PC + 8, "max pc=0x%X" % max(pcs_all))
    check("T4e 最後 16 個 cycle 都在 spin 迴圈內", set(pcs_all[-16:]) <= SPIN_PCS)
    regs = [0] * 32
    for rd, v in got: regs[rd] = v
    diff = [(i, hex(regs[i]), hex(EXP_REGS[i])) for i in range(1, 32) if regs[i] != EXP_REGS[i]]
    check("T4f 由 trace 重建的暫存器 x1~x31 與黃金模型一致", not diff, "差異(reg,實際,預期)=" + str(diff[:4]))

    arm_and_run(); time.sleep(0.01)
    rows2 = read_trace()
    check("T5 重跑後逐 cycle trace 完全相同", rows1 == rows2)

    cpu_reset(); time.sleep(0.001)
    check("T6a 執行後 reset：PC 回到 0", mmio.read(PC) == 0)
    arm_and_run(); time.sleep(0.01)
    rows3 = read_trace()
    check("T6b reset 後重新執行，trace 與第一次相同", rows3 == rows1)


In [ ]:
# ⑥ Zbb + MUL 逐指令驗證：每個指令的硬體結果 vs 黃金模型
ZBB_NAMES = set(ZBB_OPS) | {"rori", "mul"}   # 逐指令驗證：Zbb 全部 + RV32M MUL
ZBB_TABLE = []                      # (助記符, 指令PC, rd, 預期, 實際, 是否一致)
if HAS_TRACE:
    for k, (ipc, name, rd, val) in enumerate(EXP_INFO):
        if name in ZBB_NAMES:
            g = got[k] if k < len(got) else None
            ZBB_TABLE.append((name, ipc, rd, val, g[1] if g and g[0] == rd else None, bool(g and g == (rd, val))))
    by_op = {}
    for name, ipc, rd, exp, act, ok in ZBB_TABLE:
        by_op.setdefault(name, []).append(ok)
    for name in sorted(by_op):
        check("T7 %s %-7s（%d 筆）" % ("M  " if name == "mul" else "Zbb", name, len(by_op[name])), all(by_op[name]))
    missing = sorted(ZBB_NAMES - set(by_op))
    if missing: print("（本程式未涵蓋：%s）" % ", ".join(missing))
    bad = [t for t in ZBB_TABLE if not t[5]]
    if bad:
        print("不一致的 Zbb 指令：")
        for name, ipc, rd, exp, act, _ in bad:
            print("  pc=0x%X %-7s x%d 預期=0x%08X 實際=%s" % (ipc, name, rd, exp, "None" if act is None else "0x%08X" % act))


In [ ]:
# ⑦ illegal 指令偵測（需要含 illegal 暫存器的 bitstream；舊版自動略過）
feat = mmio.read(R_FEAT)
HAS_ILL = bool(feat & 1)
HAS_MUL = bool(feat & 2)
if not HAS_MUL:
    print("[WARN] bitstream 未宣告 MUL（feature=0x%X）：此版程式含 mul，請確認 bitstream 是新版" % feat)
if not HAS_ILL:
    print("[SKIP] bitstream 沒有 illegal 暫存器（feature=0x%X）：略過 T8" % feat)
else:
    cpu_reset(); time.sleep(0.001)
    check("T8a reset 狀態：illegal 旗標為 0", mmio.read(R_ILL) == 0 and mmio.read(R_ILLPC) == 0,
          "flag=%d pc=0x%X" % (mmio.read(R_ILL), mmio.read(R_ILLPC)))
    cpu_run(); time.sleep(0.01)
    check("T8b 執行後 illegal 旗標被設起（sticky）", mmio.read(R_ILL) == 1)
    check("T8c 回報第一條 illegal 的 PC = 0x%X" % EXP_ILLEGAL[0], mmio.read(R_ILLPC) == EXP_ILLEGAL[0],
          "實際 0x%X" % mmio.read(R_ILLPC))
    time.sleep(0.01)
    check("T8d 旗標維持（sticky），PC 不被後續 illegal 覆蓋", mmio.read(R_ILL) == 1 and mmio.read(R_ILLPC) == EXP_ILLEGAL[0])
    cpu_reset(); time.sleep(0.001)
    check("T8e reset 後旗標清除", mmio.read(R_ILL) == 0 and mmio.read(R_ILLPC) == 0)
    cpu_run(); time.sleep(0.01)
    check("T8f 重跑後再次回報相同 PC", mmio.read(R_ILL) == 1 and mmio.read(R_ILLPC) == EXP_ILLEGAL[0])


In [ ]:
# ⑧ 結果總表與報告檔
print("=" * 64)
for name, ok, detail in RESULTS:
    print(("PASS  " if ok else "FAIL  ") + name)
n_pass = sum(1 for _, ok, _ in RESULTS if ok)
print("=" * 64)
print("總計：%d / %d 通過  →  %s" % (n_pass, len(RESULTS), "ALL PASS ✅" if n_pass == len(RESULTS) else "有失敗項目 ❌"))

if rows1:
    print("\n前 40 個 cycle（cyc / pc / wb_data / wb_rd / wb_we）：")
    for r in rows1[:40]:
        print("%3d  0x%08X  0x%08X  x%-2d  %d" % r)
if ZBB_TABLE:
    print("\nZbb 結果（指令 PC / 助記符 / rd / 預期 / 實際）：")
    for name, ipc, rd, exp, act, ok in ZBB_TABLE:
        print("  0x%03X %-7s x%-2d 0x%08X  %s  %s" % (ipc, name, rd, exp, "0x%08X" % act if act is not None else "None      ", "OK" if ok else "<<<< 不一致"))

report = {"time": time.strftime("%Y-%m-%d %H:%M:%S"),
          "program_words": len(PROGRAM),
          "results": [{"test": n, "pass": ok, "detail": d} for n, ok, d in RESULTS],
          "zbb": [{"op": t[0], "pc": t[1], "rd": t[2], "expected": t[3], "actual": t[4], "ok": t[5]} for t in ZBB_TABLE],
          "illegal": {"present": HAS_ILL, "expected_first_pc": EXP_ILLEGAL[0] if EXP_ILLEGAL else None,
                      "expected_count": len(EXP_ILLEGAL)},
          "trace": rows1}
with open("/home/xilinx/jupyter_notebooks/cpu_test_report.json", "w") as f:
    json.dump(report, f, indent=1)
print("\n報告已存到 cpu_test_report.json")
